In [ ]:
# === Webcam capture-rate check (paste into a Colab cell; no GPU needed) ===
#
# QUESTION
#   MasterScript.py writes every webcam video with a NOMINAL speed of 30 fps
#   (cv2.VideoWriter(..., 30.0, ...)), but it saves frames as fast as the
#   camera delivers them. If the MacBook camera really delivered fewer than
#   30 frames per second (macOS cameras often slow down in dim light), the
#   video file would still SAY 30 fps, ffprobe would still report 30, but
#   frame number / 30 would no longer equal real seconds. The pipeline turns
#   frame numbers into seconds with  frame / 30  (ACTUAL_FPS), so this would
#   put the face data on the wrong time axis relative to the heartbeats.
#
# HOW THIS CHECKS IT (using only files the experiment already saved)
#   real recording duration = (wall-clock time the questionnaire phase began)
#                             - (wall-clock time the camera started)
#   effective fps           = number of frames in the .avi / real duration
#   Both wall-clock times come from the saved PPI file and camera-timing file.
#   The questionnaire phase begins the moment recording stops, so this is the
#   true length of the recording.
#
# OUTPUT
#   capture_fps_check.csv  (one row per trial) + a printed summary.
#   drift_at_end_sec = how many seconds the "frame / 30" clock is off by the
#   END of the trial (0 means the 30 fps assumption is exact).
import os
import subprocess
import numpy as np
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/ThesisExperiment_cleann"
PARTICIPANTS = [1, 3, 4, 5, 6]
ROBOTS = ["Robot1", "Robot2", "Robot3", "Robot4", "Human"]
NOMINAL_FPS = 30.0


def count_frames(avi_path):
    """Number of video frames stored in the file (no decoding, fast)."""
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-count_packets", "-select_streams", "v:0",
         "-show_entries", "stream=nb_read_packets", "-of", "csv=p=0", avi_path],
        capture_output=True, text=True)
    try:
        return int(out.stdout.strip().split(",")[0])
    except Exception:
        return np.nan


def run(base_dir, participants):
    rows = []
    for pid in participants:
        ppi_path = os.path.join(base_dir, f"participant_{pid}_continuous_ppi.csv")
        tim_path = os.path.join(base_dir, f"participant_{pid}_camera_timing.csv")
        if not (os.path.exists(ppi_path) and os.path.exists(tim_path)):
            print(f"[!] participant {pid}: PPI or camera-timing file missing, skipped.")
            continue
        ppi = pd.read_csv(ppi_path)
        timing = pd.read_csv(tim_path).drop_duplicates("robot_name", keep="last")
        cam_start = dict(zip(timing["robot_name"], timing["camera_start_time"]))
        trial_col = "trial_name"

        for robot in ROBOTS:
            avi = os.path.join(base_dir, f"participant_{pid}_{robot}_cam.avi")
            row = {"participant": pid, "robot": robot}
            q = ppi[(ppi[trial_col] == robot) & (ppi["phase"] == "questionnaire")]
            if not os.path.exists(avi) or q.empty or robot not in cam_start:
                row["note"] = "missing avi, questionnaire rows or camera start"
                rows.append(row)
                continue
            # elapsed_trial_sec restarts at each phase, so (absolute - elapsed)
            # is the exact wall-clock moment the questionnaire phase began.
            q_start = (q["absolute_time"] - q["elapsed_trial_sec"]).median()
            duration = q_start - cam_start[robot]
            frames = count_frames(avi)
            eff = frames / duration if duration > 0 else np.nan
            row.update({
                "frames_in_avi": frames,
                "real_duration_sec": round(duration, 2),
                "effective_fps": round(eff, 2),
                "duration_if_30fps_sec": round(frames / NOMINAL_FPS, 2),
                "drift_at_end_sec": round(duration - frames / NOMINAL_FPS, 2),
                "note": "",
            })
            rows.append(row)

    res = pd.DataFrame(rows)
    pd.set_option("display.width", 200)
    print(res.to_string(index=False))
    ok = res.dropna(subset=["effective_fps"])
    if len(ok):
        print("\n" + "=" * 70)
        print(f"Trials checked: {len(ok)} of {len(res)}")
        print(f"Effective fps: min {ok['effective_fps'].min():.2f}, "
              f"median {ok['effective_fps'].median():.2f}, max {ok['effective_fps'].max():.2f}")
        print(f"Drift of the frame/30 clock by end of trial: "
              f"min {ok['drift_at_end_sec'].min():.2f}s, "
              f"max {ok['drift_at_end_sec'].max():.2f}s")
        print("\nHow to read this: if effective fps is close to 30 in every trial and "
              "the drift is small (well under 1 second, the fusion tolerance), the "
              "frame/30 timestamps are trustworthy. A clearly lower fps means face "
              "timestamps need to be rescaled with the measured rate.")
    res.to_csv(os.path.join(base_dir, "capture_fps_check.csv"), index=False)
    return res


if __name__ == "__main__":
    from google.colab import drive
    drive.mount("/content/drive")
    run(BASE_DIR, PARTICIPANTS)